# Ch.1 — Recommender Systems Fundamentals _(Exercise)_

> **The story.** In **1992**, researchers at **Xerox PARC** built **Tapestry** — the first system ever to use the phrase "collaborative filtering." The idea was deceptively simple: let users annotate email messages, then route future messages based on *other people's* annotations. No algorithm had done this before. Three years later, the **GroupLens** team at the University of Minnesota applied the same principle to Usenet news articles, coined the term *recommender system*, and released the **MovieLens** dataset that would become the Rosetta Stone of recommendation research for the next 30 years. In **1998**, Amazon engineer **Greg Linden** filed a patent for **item-based collaborative filtering** — "customers who bought X also bought Y" — a system so effective it now drives 35% of Amazon's revenue. In **2006**, Netflix launched the **Netflix Prize**: $1,000,000 to anyone who could improve their CineMatch recommendation algorithm by 10%. Forty-four thousand teams from 186 countries competed for three years. The winners, **BellKor's Pragmatic Chaos**, crossed the line in 2009 with a 10.06% improvement — and proved, for the first time with rigorous evidence, that recommendation quality has direct, measurable, seven-figure business value. In **2015**, Spotify launched **Discover Weekly** — a personalised 30-song playlist refreshed every Monday, built entirely on collaborative filtering of listening histories. Within its first year, users streamed Discover Weekly songs over 5 billion times.
>
> **Where you are.** This is chapter one of the Recommender Systems track. You are the lead ML engineer at **FlixAI**, a streaming platform competing with Netflix and Disney+. Your first task is the simplest possible model: recommend the most popular movies to everyone. It sounds lazy — and it is — but it establishes the entire evaluation framework, the metrics vocabulary, and the baseline numbers that every subsequent chapter must beat. The Recommender Systems track teaches the progression from naive popularity ranking (this chapter) through neighbourhood models (Ch.2), matrix factorization (Ch.3), deep learning (Ch.4–5), and production serving (Ch.6). Every chapter inherits the metrics and vocabulary introduced here.
>
> **Notation.** $m$ — number of users (943); $n$ — number of items (1,682); $R \in \mathbb{R}^{m \times n}$ — user-item rating matrix; $r_{ui}$ — rating by user $u$ on item $i$ (1–5 or missing); $\hat{r}_{ui}$ — predicted score; $K$ — recommendation list size (10); $\text{HR}@K$ — hit rate at $K$ (fraction of users with ≥1 relevant item in top-$K$); $\text{NDCG}@K$ — position-weighted accuracy; $\mu$ — global mean rating (3.53); $C$ — Bayesian damping constant (median rating count per item).

---

## 0 · The Challenge

> **The mission**: FlixAI — build a production movie recommendation engine. Five constraints must be satisfied: >85% HR@10, cold start for new users/items, scalability to 1M+ ratings, diversity beyond popular movies, and explainable recommendations.

**What we know so far:**
- MovieLens 100k: 943 users, 1,682 movies, 100,000 explicit ratings (scale 1–5)
- Business context: 60% of user sessions end without clicking a single recommendation → churn
- CEO mandate: "Netflix recommendations drive 80% of viewing hours; ours drive 12% — fix it"
- **No model. No metrics. No baseline.**

**What's blocking us:**
The VP of Product won't accept "it feels better." You need falsifiable numbers: "Our system achieves X% HR@10, which beats the naive baseline by Y points." Right now you have none of those numbers. Before building matrix factorization or deep learning, you must answer: *How do you measure recommendation quality?* and *What accuracy can you achieve with zero ML?*

**What this chapter unlocks:**
- **Evaluation framework**: HR@K, NDCG@K, MRR, Coverage — the shared language for Ch.1–6
- **Popularity baseline**: ~42% HR@10 using Bayesian-averaged top movies
- **Gap quantified**: 50 percentage points to the 85% target — the arc of the entire track

| Section | Content |
| --- | --- |
| § 1 Core Idea | What a recommender is; three families of approaches |
| § 2 Running Example | MovieLens 100k dataset; sparsity problem |
| § 3 Evaluation Framework | HR@K, NDCG@K, leave-one-out split |
| § 4 Popularity Baseline | Bayesian average; top-10 results |

## 1 · The Core Idea

Before you can ask "which 10 movies should I show this user?", you need to know what counts as success. That's the first job of this chapter. The second is to build something embarrassingly simple — the popularity baseline — so you have a floor to beat and a story to tell about why it falls short.

Three families of recommenders exist, and FlixAI will eventually use all three:

1. **Content-Based**: Recommend items similar to what you liked (based on item features — genre, director, cast)
2. **Collaborative Filtering**: Recommend items that similar users liked — no item knowledge required
3. **Hybrid**: Combine both signals; this is what Ch.5 builds

This chapter builds the simplest approach — the **popularity baseline** — and establishes the evaluation metrics (HR@K, NDCG@K, MRR, Coverage) used throughout the entire track.

> **Optional depth:** Formal treatment of ranking metrics and their properties: see [MathUnderTheHood ch06](../../00-math-under-the-hood/ch06).

## 2 · Running Example

You are a data scientist at **FlixAI**, a movie streaming platform. The VP of Product calls you in: "Netflix recommendations drive 80% of their viewing hours. Ours drive 12%. I need a recommendation widget on our homepage by Friday." The dataset you have is **MovieLens 100k** — 100,000 explicit ratings from 943 users on 1,682 movies, collected between September 1997 and April 1998 by the GroupLens research group. Ratings are integers from 1 (terrible) to 5 (loved it).

There are three facts that define the difficulty of what you are trying to do:

1. **Sparsity**: 93.7% of the 943×1,682 rating matrix is empty. A missing entry does not mean the user dislikes the movie — it means they have not seen it.
2. **Scale**: 943 × 1,682 = 1,585,726 possible user-movie pairs, but only 100,000 are observed.
3. **Benchmark gap**: A random recommender (pick 10 movies uniformly at random) achieves HR@10 ≈ **0.6%**. Your target is **>85%**. That 84-point gap is the arc of the entire track.

```mermaid
graph LR
    A["100k Users\n× 1,700 Movies\n= 170M cells"] --> B["Only 100k\nratings exist\n= 99.94% empty"]
    B --> C["Sparsity problem:\ncan't recommend\nwhat's not rated"]
    C --> D["Solution:\ninfer missing\nfrom patterns"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Imports
# 2. Compute `SEED` using `set_theme()`
# 3. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Load MovieLens 100k)
#
# Steps:
# 1. Load MovieLens 100k
# 2. Compute `ratings` using `read_csv()`
# 3. Compute `movies` using `read_csv()`
# 4. Call `nunique()` to produce the result
#
# Hint:
#    ratings = pd.read_csv(???)
#    movies = pd.read_csv(???)

In [ ]:
# TODO: Implement this cell
#  (Exploratory Data Analysis)
#
# Steps:
# 1. Exploratory Data Analysis
# 2. Plot results -- call `hist()`
# 3. Plot results -- call `median()`
# 4. Plot results -- call `suptitle()`
# 5. Call `mean()` to produce the result
#
# Hint:
#    axes = plt.subplots(???)
#    user_counts = ratings.groupby(???)
#    item_counts = ratings.groupby(???)

## 3 · Evaluation Framework

Every claim you make about a recommender — "this model is better" — needs a number behind it. The numbers come from a shared evaluation protocol: hold out one rating per user as the test item, build a top-K recommendation list, and check whether the test item appears in it. This section defines the four metrics FlixAI uses throughout all chapters.

We use **leave-one-out** evaluation: for each user, hold out the **last** rating (by timestamp) as the test item. The model must rank this item in the top-K among 99 random negatives + 1 positive.

**Key Metrics:**
- **Hit Rate@K**: Fraction of users where the test item appears in top-K — the primary FlixAI metric
- **NDCG@K**: Normalized Discounted Cumulative Gain — rewards items ranked higher over lower
- **Precision@K**: Fraction of the top-K list that is relevant (useful when multiple items are relevant)

> **Optional depth:** The choice of 99 random negatives follows the standard BPR evaluation protocol from Rendle et al. (2009). With 1,682 items, full ranking is feasible for this dataset but computationally expensive at scale. The 99-negative protocol approximates full ranking while being 16× faster.

In [ ]:
def leave_one_out_split(ratings_df):
    """
    TODO #4: Implement `leave_one_out_split()`.

    Steps:
    1. Leave-One-Out Train/Test Split
    2. Call `ratings()` to produce the result

    Hint:
    ratings_sorted = ratings_df.sort_values(???)
    test = ratings_sorted.groupby(???)
    train = ratings_sorted.drop(???)

    Returns: train, test
    """
    raise NotImplementedError("TODO: implement leave_one_out_split()")

In [ ]:
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #5: Implement `hit_rate_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement hit_rate_at_k()")

def ndcg_at_k(test_df, top_k_per_user, k=10):
    """
    TODO #5: Implement `ndcg_at_k()`.

    Steps:
    1. Evaluation Metrics
    2. Define helper function `ndcg_at_k()`
    3. Process data

    Hint:
    recs = top_k_per_user.get(???)
    rank = recs.index(???)

    Returns: hits / len(test_df)
    """
    raise NotImplementedError("TODO: implement ndcg_at_k()")

**Reflection — § 3 Evaluation Framework**

The evaluation protocol is doing something subtle: by using 99 *random* negatives instead of ranking all 1,682 items, it gives every model a slight advantage over real production conditions. In production, the model must rank the test item against every unrated movie — not just 99 sampled ones. That said, the random-negative protocol is the field standard for MovieLens 100k comparisons, so all HR@10 numbers in this track are directly comparable to published benchmarks.

What this section fixed: you can now state "our model achieves X% HR@10" and defend the number. What it doesn't fix: you have no model yet — only an evaluation rig. The next section builds the first model.

## 4 · Popularity Baseline

The naive approach: count how many times each movie was rated and recommend the most-rated ones. The problem is that a movie with 5 ratings averaging 5.0 beats a movie with 500 ratings averaging 4.5, which is statistically unreliable. You need a score that shrinks toward the global mean when evidence is thin. That is what the Bayesian average does — it adds $C$ imaginary ratings at the global mean $\mu$ before computing the average, so low-count movies regress to the mean rather than floating to the top.

The score for item $i$ is:

$$\text{score}(i) = \frac{|U_i| \cdot \bar{r}_i + C \cdot \mu}{|U_i| + C}$$

where $|U_i|$ is the rating count, $\bar{r}_i$ is the raw average, $\mu$ is the global mean, and $C$ is set to the median rating count per item. As $|U_i| \to \infty$, the Bayesian average converges to the raw average — once you have enough data, the prior washes out.

> **Optional depth:** This is a conjugate Bayesian model with a Gaussian likelihood and known variance. The posterior mean given prior $\mathcal{N}(\mu, \sigma^2/C)$ and $n$ observations is exactly the Bayesian average formula above. See [MathUnderTheHood ch04](../../00-math-under-the-hood/ch04) for the derivation.

In [ ]:
# TODO: Implement this cell
#  (Popularity Baseline)
#
# Steps:
# 1. Popularity Baseline
# 2. Compute `item_stats["bayesian_avg"]`
# 3. Compute `top_items_sorted` using `sort_values()`
# 4. Aggregate data into `user_rated_train` -- use `groupby()`
# 5. Compute `top_k_pop` using `unique()`
# 6. Compute `hr` using `hit_rate_at_k()`
# 7. Process data
#
# Hint:
#    item_stats = train.groupby(???)
#    top_items_sorted = item_stats.sort_values(???)
#    user_rated_train = train.groupby(???)
#    rated = user_rated_train.get(???)

In [ ]:
# TODO: Implement this cell
#  (Top-10 Most Popular Movies)
#
# Steps:
# 1. Top-10 Most Popular Movies
# 2. Call `iterrows()` to produce the result
#
# Hint:
#    top_10 = item_stats.sort_values(???)
#    top_10_with_titles = top_10.merge(???)

In [ ]:
# TODO: Implement this cell
#  (Visualise: Bayesian Average vs Raw Average)
#
# Steps:
# 1. Visualise: Bayesian Average vs Raw Average
# 2. Plot results -- call `scatter()`
#
# Hint:
#    ax = plt.subplots(???)
#    scatter = ax.scatter(???)

In [ ]:
# TODO: Implement this cell
#  (HR@k Curve: How does hit rate change with k?)
#
# Steps:
# 1. HR@k Curve: How does hit rate change with k?
# 2. Call `unique()` to produce the result
# 3. Plot results -- call `subplots()`
# 4. Process data
#
# Hint:
#    rated = user_rated_train.get(???)
#    ax = plt.subplots(???)

In [ ]:
# TODO: Implement this cell
#  (Genre Distribution of Top-10 Recommendations)
#
# Steps:
# 1. Genre Distribution of Top-10 Recommendations
# 2. Compute `top_10_ids` using `sort_values()`
# 3. Plot results -- call `subplots()`
# 4. Process data
#
# Hint:
#    top_10_ids = item_stats.sort_values(???)
#    ax = plt.subplots(???)

**Reflection — § 4 Popularity Baseline**

The popularity baseline achieved ~42% HR@10 — 70× better than a random recommender (0.6%), but still 43 points short of the 85% target. Three structural failures explain the gap:

1. **No personalisation**: User 1 (loves horror) and User 2 (loves romance) receive identical top-10 lists. The model reads zero signal from user history.
2. **Diversity collapse**: The top-10 is dominated by Drama. Every user gets the same genre profile, regardless of taste.
3. **No cold-start problem yet** — but that's because this model ignores ratings entirely for recommendation (it only uses aggregate counts). That will become a constraint to solve in Ch.5–6.

What this section fixed: you have a measurable, reproducible floor. Every algorithm in Ch.2–6 is evaluated against this 42% baseline. What it exposed: personalisation is the entire remaining gap.

## Summary

**What this chapter unlocked:**

| # | Constraint | Target | Ch.1 Result |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | **42%** — floor established |
| 2 | COLD START | New users/items | Same list for everyone (not solved) |
| 3 | SCALABILITY | 1M+ ratings | Pre-computed list (solved for this size) |
| 4 | DIVERSITY | Not just popular | Drama-dominated (not solved) |
| 5 | EXPLAINABILITY | "Because you liked X" | "Because it's popular" (partial) |

**Key takeaways:**
- Bayesian damping prevents low-count movies from gaming the leaderboard; set $C$ = median rating count
- HR@10 is the primary FlixAI metric; it measures whether the system surfaces at least one relevant film per user
- The 99-negative evaluation protocol is the field standard for MovieLens 100k comparisons — all track numbers are directly comparable
- Popularity is not personalisation; it is the floor, not the ceiling

**Checkpoint:** Popularity baseline reaches 42% HR@10 on MovieLens 100k. The evaluation framework — HR@K, NDCG@K, leave-one-out split — is established and shared across all six chapters. 43 percentage points remain to the 85% FlixAI target.

**Forward:** Ch.2 introduces collaborative filtering — the first time different users receive different recommendations. You will see HR@10 jump to ~68% from personalisation alone, then hit a sparsity ceiling that forces a more powerful representation in Ch.3.

## Exercises

**Exercise 1 — Raw Average vs Bayesian Average**
Build a popularity baseline using raw average rating (no Bayesian damping). Compare HR@10 against the Bayesian baseline. Which is better and why?

**Exercise 2 — Rating Count Baseline**
Instead of average rating, rank movies by total number of ratings (most-rated = most popular). Compare HR@10. Does "most rated" outperform "highest rated"?

**Exercise 3 — Genre-Specific Popularity**
Build a popularity baseline that's genre-aware: for users whose most-rated genre is Sci-Fi, recommend the top Sci-Fi movies. Compare HR@10 and diversity against the global popularity baseline.

In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold — Raw Average vs Bayesian)
#
# Steps:
# 1. Set up: Exercise 1 scaffold — Raw Average vs Bayesian
# 2. Process data
#
# Hint:
#    raw_top_items = item_stats.sort_values(???)

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold — Rating Count Baseline)
#
# Steps:
# 1. Set up: Exercise 2 scaffold — Rating Count Baseline
# 2. Process data
#
# Hint:
#    count_top_items = item_stats.sort_values(???)

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold — Genre-Specific Popularity)
#
# Steps:
# 1. Set up: Exercise 3 scaffold — Genre-Specific Popularity
# 2. Process data
#
# Hint:
#    # implement using the APIs described above